# Kev-4B vs Kev-9B — 기존 실험 결과 감사 (GPU 불필요)

지난 Colab에서 저장된 BugsInPy 100건을 **재실행하지 않고** 점검합니다. Google Drive 읽기만으로 4B/9B의 전체 L1/L2/L3 확률 벡터를 비교하고, 입력 해시·누락·중복·로그에 남은 체크포인트 이름을 살펴봅니다.

**한계:** 기존 결과 파일의 `model_label`은 당시 실행 코드가 부여한 이름입니다. 기존 실행에는 `/v1/models.run`으로 확인한 모델 식별 정보가 없으므로, 로그에 4B/9B가 언급돼도 실제 각 예측이 해당 체크포인트에서 생성됐음을 소급 증명할 수 없습니다. 결과의 수학적 일치 여부만 먼저 확인할 수 있습니다.

사용법: 아래 두 코드 셀을 순서대로 실행합니다. **런타임 GPU는 필요하지 않습니다** (CPU 가능). 결과는 원래 실행 폴더의 `outputs/bugsinpy_legacy_audit.json`으로 저장됩니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")
RESULTS_ROOT = Path("/content/drive/MyDrive/adaptive-llm-router/results")
# Optional: set the exact prior RUN_ID from your previous experiment, e.g.
# "kev_a100_20261026_...". Blank automatically picks newest 100-case run.
TARGET_RUN_ID = ""
if not RESULTS_ROOT.is_dir():
    raise RuntimeError("Google Drive results folder not found; check your Google account")

folders = [p for p in RESULTS_ROOT.glob("kev_a100_*")
           if p.is_dir()
           and (p / "outputs" / "bugsinpy_kev_decisions.jsonl").is_file()]
if TARGET_RUN_ID:
    RUN_ROOT = RESULTS_ROOT / TARGET_RUN_ID
    if RUN_ROOT not in folders:
        raise FileNotFoundError("Specified run has no saved BugsInPy results: " + TARGET_RUN_ID)
else:
    def size_of_run(p):
        status = p / "outputs" / "bugsinpy_comparison.json"
        try:
            return int(json.loads(status.read_text()).get("n_paired") or 0)
        except (OSError, ValueError, TypeError):
            return 0
    completed_100 = [p for p in folders if size_of_run(p) >= 100]
    selection = completed_100 or folders
    if not selection:
        raise FileNotFoundError("No existing BugsInPy results found in MyDrive/results")
    RUN_ROOT = max(selection, key=lambda p: p.name)
print("Selected previous run:", RUN_ROOT.name)
print("Found:", (RUN_ROOT / "outputs/bugsinpy_kev_decisions.jsonl").exists())
print("GPU required: NO")

In [ ]:
import subprocess, sys, importlib
from pathlib import Path
ROOT_AUDIT = Path("/content/adaptive-llm-router-audit")
if not (ROOT_AUDIT / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--quiet",
         "https://github.com/namyikim/adaptive-llm-router.git", str(ROOT_AUDIT)],
        check=True
    )
else:
    subprocess.run(["git", "-C", str(ROOT_AUDIT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(ROOT_AUDIT / "src"))
importlib.invalidate_caches()
if "adaptive_llm_router.legacy_result_audit" in sys.modules:
    importlib.reload(sys.modules["adaptive_llm_router.legacy_result_audit"])
from adaptive_llm_router.legacy_result_audit import audit_folder
report = audit_folder(RUN_ROOT)
report_path = RUN_ROOT / "outputs" / "bugsinpy_legacy_audit.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n",
                       encoding="utf-8")
print("\n=== Previous-run audit ===")
for key in (
    "status", "n_expected_cases", "n_valid_pairs",
    "n_invalid_or_mismatched_rows", "n_duplicate_model_case_keys",
    "n_raw_tier_matches", "n_final_tier_matches",
    "n_exact_pmax_matches", "n_exact_probability_vector_matches",
    "n_both_checkpoint_identities_attested",
    "mean_max_abs_probability_diff", "max_abs_probability_diff",
):
    print(f"{key}: {report[key]}")
print("Flags:", ", ".join(report["flags"]) or "(none)")
print("Historical server log mentions:", report["requested_checkpoint_mentions_in_log"])
print("\nPer-model:", json.dumps(report["models"], ensure_ascii=False, indent=2))
print("\nSaved report:", report_path)
print("You can share this audit JSON here without sharing raw test source text.")